# Classification Efficiency Metrics

Profiles the computational cost of every exit of the early-exit ResNet18, VGG16 and MobileNetV2 classifiers, with and without the Upfront Exit Predictor (UEP): parameters, GFLOPs, latency, peak GPU memory and GPU energy per image.

Weights and inputs are random (one 224x224 image, batch size 1), so no dataset or trained checkpoint is needed.

## 1. Setup

In [24]:
!pip -q install fvcore pandas nvidia-ml-py

In [25]:
import os
import time
import threading
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models

from torchvision.models import ResNet18_Weights, VGG16_Weights, MobileNet_V2_Weights

from fvcore.nn import FlopCountAnalysis

# =========================================================
# Reproducibility
# =========================================================

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================================
# Device
# =========================================================

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

    # Let cuDNN select the fastest kernels for the fixed input shape
    torch.backends.cudnn.benchmark = True

# =========================================================
# Input configuration
# =========================================================

IMAGE_SIZE = 224
NUM_CLASSES = 10

# Latency is measured per single image
BATCH_SIZE = 1

# Random input image; no dataset is needed for cost profiling
random_input = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)

print("Random input shape:", tuple(random_input.shape))

# =========================================================
# Benchmark configuration
# =========================================================

WARMUP_ITERS = 30
TIMED_ITERS = 100

# NVML power sampling interval (seconds)
POWER_SAMPLE_INTERVAL = 0.01

# =========================================================
# NVML setup for energy measurement
# =========================================================

NVML_AVAILABLE = False
NVML_HANDLE = None

if DEVICE.type == "cuda":
    try:
        import pynvml
        pynvml.nvmlInit()
        NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())
        NVML_AVAILABLE = True
        print("NVML: available")
    except Exception as e:
        print("NVML not available. Energy measurements will be skipped.")
        print("Reason:", e)

PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
Device: cuda:0
GPU: Tesla T4
Random input shape: (1, 3, 224, 224)
NVML: available


## 2. Early-exit models

Each backbone is split into a shared feature extractor, backbone segments and exit heads. ResNet18 and MobileNetV2 have two exits; VGG16 has three. `forward` returns the output of every exit and the shared features used by the UEP.

In [26]:
class EarlyExitResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Random weights; only computational cost is measured
        self.resnet = models.resnet18(weights=None)
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, num_classes)

        # Shared feature extractor
        self.feature_extractor = nn.Sequential(
            self.resnet.conv1,
            self.resnet.bn1,
            self.resnet.relu,
            self.resnet.maxpool,
            self.resnet.layer1,
        )

        # Backbone segment up to Exit 1 (layer2 + layer3)
        self.block1 = nn.Sequential(self.resnet.layer2, self.resnet.layer3)

        # Backbone segment up to the final exit (layer4)
        self.block2 = self.resnet.layer4

        # Exit 1 head
        self.exit1 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, num_classes)
        )

        # Final exit head
        self.final_classifier = nn.Sequential(self.resnet.avgpool, nn.Flatten(), self.resnet.fc)

    def forward(self, x):
        features_for_predictor = self.feature_extractor(x)
        x1 = self.block1(features_for_predictor)
        exit1_out = self.exit1(x1)
        x2 = self.block2(x1)
        final_out = self.final_classifier(x2)
        return [exit1_out, final_out], features_for_predictor

In [27]:
class EarlyExitVGG(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Random weights; only computational cost is measured
        self.vgg = models.vgg16(weights=None)
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Linear(num_ftrs, num_classes)

        # Shared feature extractor
        self.feature_extractor = self.vgg.features[:10]

        # Backbone segments up to Exit 1 and Exit 2
        self.block1 = self.vgg.features[10:17]
        self.block2 = self.vgg.features[17:24]

        # Backbone segment up to the final exit
        self.block3 = self.vgg.features[24:]

        # Exit 1 head
        self.exit1 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256, num_classes)
        )

        # Exit 2 head
        self.exit2 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, num_classes)
        )

        # Final exit head
        self.final_classifier = nn.Sequential(self.vgg.avgpool, nn.Flatten(), self.vgg.classifier)

    def forward(self, x):
        features_for_predictor = self.feature_extractor(x)
        x1 = self.block1(features_for_predictor)
        exit1_out = self.exit1(x1)
        x2 = self.block2(x1)
        exit2_out = self.exit2(x2)
        x3 = self.block3(x2)
        final_out = self.final_classifier(x3)
        return [exit1_out, exit2_out, final_out], features_for_predictor

In [28]:
class EarlyExitMobileNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Random weights; only computational cost is measured
        self.mobilenet = models.mobilenet_v2(weights=None)
        num_ftrs = self.mobilenet.classifier[1].in_features
        self.mobilenet.classifier[1] = nn.Linear(num_ftrs, num_classes)

        # Shared feature extractor
        self.feature_extractor = self.mobilenet.features[:4]

        # Backbone segment up to Exit 1
        self.block1 = self.mobilenet.features[4:14]

        # Backbone segment up to the final exit
        self.block2 = self.mobilenet.features[14:]

        # Exit 1 head
        self.exit1 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(96, num_classes)
        )

        # Final exit head
        self.final_classifier = self.mobilenet.classifier

    def forward(self, x):
        features_for_predictor = self.feature_extractor(x)
        x1 = self.block1(features_for_predictor)
        exit1_out = self.exit1(x1)
        x2 = self.block2(x1)
        x2 = F.adaptive_avg_pool2d(x2, (1, 1)).reshape(x2.shape[0], -1)
        final_out = self.final_classifier(x2)
        return [exit1_out, final_out], features_for_predictor

## 3. Upfront Exit Predictor (UEP)

A lightweight CNN (two Conv-BN-ReLU-MaxPool blocks followed by a fully connected classifier) that takes the shared features and outputs one score per exit.

In [29]:
class ExitPredictorV2(nn.Module):
    def __init__(self, input_channels, spatial_dim, num_exits):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        flattened_size = 16 * (spatial_dim // 4) * (spatial_dim // 4)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_exits),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

## 4. Exit-path execution

Runs only the layers needed for the selected exit, either directly or after the UEP.

In [30]:
# =========================================================
# Execute the network up to a selected exit
# =========================================================


def get_model_type(model):
    """
    Identify architecture without relying on isinstance().
    This avoids Jupyter class-redefinition issues.
    """
    if hasattr(model, "resnet"):
        return "ResNet18"
    elif hasattr(model, "vgg"):
        return "VGG16"
    elif hasattr(model, "mobilenet"):
        return "MobileNetV2"
    else:
        raise TypeError(f"Unknown model architecture: {type(model)}")


def forward_to_exit(model, x, exit_idx):
    """
    Execute ONLY the selected exit path.

    exit_idx:
        0 = earliest exit
        final index = final exit
    """
    model_type = get_model_type(model)

    # =====================================================
    # ResNet18
    # =====================================================
    if model_type == "ResNet18":
        features = model.feature_extractor(x)
        x1 = model.block1(features)
        if exit_idx == 0:
            return model.exit1(x1)
        elif exit_idx == 1:
            x2 = model.block2(x1)
            return model.final_classifier(x2)

    # =====================================================
    # VGG16
    # =====================================================
    elif model_type == "VGG16":
        features = model.feature_extractor(x)
        x1 = model.block1(features)
        if exit_idx == 0:
            return model.exit1(x1)
        x2 = model.block2(x1)
        if exit_idx == 1:
            return model.exit2(x2)
        elif exit_idx == 2:
            x3 = model.block3(x2)
            return model.final_classifier(x3)

    # =====================================================
    # MobileNetV2
    # =====================================================
    elif model_type == "MobileNetV2":
        features = model.feature_extractor(x)
        x1 = model.block1(features)
        if exit_idx == 0:
            return model.exit1(x1)
        elif exit_idx == 1:
            x2 = model.block2(x1)
            x2 = F.adaptive_avg_pool2d(x2, (1, 1)).reshape(x2.shape[0], -1)
            return model.final_classifier(x2)
    raise ValueError(f"Invalid exit index {exit_idx} for {model_type}")


def forward_from_features(model, features, exit_idx):
    """
    Continue execution from the shared feature extractor.
    """
    model_type = get_model_type(model)

    # =====================================================
    # ResNet18
    # =====================================================
    if model_type == "ResNet18":
        x1 = model.block1(features)
        if exit_idx == 0:
            return model.exit1(x1)
        elif exit_idx == 1:
            x2 = model.block2(x1)
            return model.final_classifier(x2)

    # =====================================================
    # VGG16
    # =====================================================
    elif model_type == "VGG16":
        x1 = model.block1(features)
        if exit_idx == 0:
            return model.exit1(x1)
        x2 = model.block2(x1)
        if exit_idx == 1:
            return model.exit2(x2)
        elif exit_idx == 2:
            x3 = model.block3(x2)
            return model.final_classifier(x3)

    # =====================================================
    # MobileNetV2
    # =====================================================
    elif model_type == "MobileNetV2":
        x1 = model.block1(features)
        if exit_idx == 0:
            return model.exit1(x1)
        elif exit_idx == 1:
            x2 = model.block2(x1)
            x2 = F.adaptive_avg_pool2d(x2, (1, 1)).reshape(x2.shape[0], -1)
            return model.final_classifier(x2)
    raise ValueError(f"Invalid exit index {exit_idx} for {model_type}")


def forward_with_uep(model, predictor, x, exit_idx):
    """
    UEP-inclusive path:

        Input
          ↓
    Shared feature extractor
          ↓
        UEP
          ↓
    Selected exit path
    """
    features = model.feature_extractor(x)

    # UEP forward pass (its output is not needed for timing)
    _ = predictor(features)

    # Continue only through selected exit
    return forward_from_features(model, features, exit_idx)

## 5. Parameter counts

Counts the total deployed parameters and the parameters evaluated along each exit path (shared modules are counted once).

In [31]:
def count_unique_parameters(modules):
    """
    Count unique parameters without double counting
    shared modules.
    """
    if not isinstance(modules, (list, tuple)):
        modules = [modules]
    seen = set()
    total = 0
    for module in modules:
        for param in module.parameters():
            pid = id(param)
            if pid not in seen:
                seen.add(pid)
                total += param.numel()
    return total


def get_exit_modules(model, exit_idx):
    # =====================================================
    # ResNet18
    # =====================================================
    if isinstance(model, EarlyExitResNet):
        if exit_idx == 0:
            return [model.feature_extractor, model.block1, model.exit1]
        if exit_idx == 1:
            return [model.feature_extractor, model.block1, model.block2, model.final_classifier]

    # =====================================================
    # VGG16
    # =====================================================
    elif isinstance(model, EarlyExitVGG):
        if exit_idx == 0:
            return [model.feature_extractor, model.block1, model.exit1]
        if exit_idx == 1:
            return [model.feature_extractor, model.block1, model.block2, model.exit2]
        if exit_idx == 2:
            return [
                model.feature_extractor,
                model.block1,
                model.block2,
                model.block3,
                model.final_classifier,
            ]

    # =====================================================
    # MobileNetV2
    # =====================================================
    elif isinstance(model, EarlyExitMobileNet):
        if exit_idx == 0:
            return [model.feature_extractor, model.block1, model.exit1]
        if exit_idx == 1:
            return [model.feature_extractor, model.block1, model.block2, model.final_classifier]
    raise ValueError(f"Invalid exit {exit_idx}")


def calculate_parameter_metrics(model, predictor, exit_idx):
    # Complete deployed system
    main_total = count_unique_parameters(model)
    predictor_total = count_unique_parameters(predictor)
    deployed_total = main_total + predictor_total

    # Parameters evaluated along this exit
    exit_modules = get_exit_modules(model, exit_idx)
    evaluated_main = count_unique_parameters(exit_modules)
    evaluated_system = evaluated_main + predictor_total
    return {
        "Main_Model_Params": main_total,
        "UEP_Params": predictor_total,
        "Total_Deployed_Params": deployed_total,
        "Evaluated_Main_Params": evaluated_main,
        "Evaluated_System_Params": evaluated_system,
        "Evaluated_Fraction_pct": 100.0 * evaluated_system / deployed_total,
    }

## 6. GFLOPs

In [32]:
# =========================================================
# GFLOPs of a single exit path
# =========================================================


def calculate_gflops(model, predictor, random_input, exit_idx, include_uep=False):
    """
    Calculate GFLOPs for batch size 1.

    include_uep=False:
        shared backbone + selected exit

    include_uep=True:
        shared backbone + UEP + selected exit
    """

    class PathWrapper(nn.Module):
        def __init__(self, model, predictor, exit_idx, include_uep):
            super().__init__()

            # Register actual modules
            self.model = model
            self.predictor = predictor
            self.exit_idx = exit_idx
            self.include_uep = include_uep

        def forward(self, x):
            if self.include_uep:
                return forward_with_uep(self.model, self.predictor, x, self.exit_idx)
            return forward_to_exit(self.model, x, self.exit_idx)

    # -----------------------------------------------------
    # Wrap the selected exit path as a module for fvcore
    # -----------------------------------------------------
    wrapper = PathWrapper(
        model=model, predictor=predictor, exit_idx=exit_idx, include_uep=include_uep
    ).to(DEVICE)
    wrapper.eval()
    random_input = random_input.to(DEVICE)

    # -----------------------------------------------------
    # Count FLOPs with fvcore
    # -----------------------------------------------------
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        flop_analysis = FlopCountAnalysis(wrapper, random_input)
        flop_analysis.unsupported_ops_warnings(False)
        flop_analysis.tracer_warnings("none")
        total_flops = flop_analysis.total()
    return total_flops / 1e9

### Sanity check

Uses `models_and_predictors`, which is created in Section 9; run that cell first.

In [33]:
# =========================================================
# Sanity check: ResNet18 Exit 1 forward pass and GFLOPs
# =========================================================

model_name = "ResNet18"

model, predictor = models_and_predictors[model_name]

print("Detected model type:")
print(get_model_type(model))

print("\nTesting direct forward...")

with torch.inference_mode():
    output = forward_to_exit(model, random_input, exit_idx=0)

print("Exit 1 output shape:", tuple(output.shape))

print("\nTesting Exit 1 GFLOPs...")

gflops_exit1 = calculate_gflops(
    model=model, predictor=predictor, random_input=random_input, exit_idx=0, include_uep=False
)

print(f"ResNet18 Exit 1 GFLOPs: {gflops_exit1:.4f}")

Detected model type:
ResNet18

Testing direct forward...
Exit 1 output shape: (1, 10)

Testing Exit 1 GFLOPs...


The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
model.final_classifier, model.final_classifier.1, model.resnet.avgpool, model.resnet.fc, model.resnet.layer4, model.resnet.layer4.0, model.resnet.layer4.0.bn1, model.resnet.layer4.0.bn2, model.resnet.layer4.0.conv1, model.resnet.layer4.0.conv2, model.resnet.layer4.0.downsample, model.resnet.layer4.0.downsample.0, model.resnet.layer4.0.downsample.1, model.resnet.layer4.0.relu, model.resnet.layer4.1, model.resnet.layer4.1.bn1, model.resnet.layer4.1.bn2, model.resnet.layer4.1.conv1, model.resnet.layer4.1.conv2, model.resnet.layer4.1.relu, predictor, predictor.conv_layers, predictor.conv_layers.0, predictor.conv_layers.1, predictor.conv_layers.2, predictor.conv_layers.3, pr

ResNet18 Exit 1 GFLOPs: 1.4073


## 7. Latency and GPU energy

Mean, median and P95 latency over the timed iterations after warm-up (CUDA events on GPU, `perf_counter` on CPU). GPU power is sampled with NVML in a background thread.

In [34]:
def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def benchmark_latency(
    inference_fn, input_tensor, warmup_iters=WARMUP_ITERS, timed_iters=TIMED_ITERS
):
    # =====================================================
    # Warm-up
    # =====================================================
    with torch.inference_mode():
        for _ in range(warmup_iters):
            inference_fn(input_tensor)
    synchronize()

    # =====================================================
    # GPU timing
    # =====================================================
    if DEVICE.type == "cuda":
        latencies_ms = []

        # ---------------------------------------------
        # NVML power sampling
        # ---------------------------------------------
        power_samples_mw = []
        stop_power_sampling = threading.Event()
        power_thread = None

        def sample_power():
            while not stop_power_sampling.is_set():
                try:
                    power_mw = pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE)
                    power_samples_mw.append(power_mw)
                except Exception:
                    pass
                time.sleep(POWER_SAMPLE_INTERVAL)

        if NVML_AVAILABLE:
            power_thread = threading.Thread(target=sample_power, daemon=True)
            power_thread.start()

        # ---------------------------------------------
        # Overall wall-clock time
        # Used for energy calculation
        # ---------------------------------------------
        overall_start = time.perf_counter()
        for _ in range(timed_iters):
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            start_event.record()
            with torch.inference_mode():
                inference_fn(input_tensor)
            end_event.record()
            end_event.synchronize()
            elapsed_ms = start_event.elapsed_time(end_event)
            latencies_ms.append(elapsed_ms)
        overall_end = time.perf_counter()

        # ---------------------------------------------
        # Stop power sampling
        # ---------------------------------------------
        if NVML_AVAILABLE:
            stop_power_sampling.set()
            if power_thread is not None:
                power_thread.join()
        latencies_ms = np.asarray(latencies_ms)
        total_wall_time_s = overall_end - overall_start

    # =====================================================
    # CPU timing
    # =====================================================
    else:
        latencies_ms = []
        overall_start = time.perf_counter()
        for _ in range(timed_iters):
            start = time.perf_counter()
            with torch.inference_mode():
                inference_fn(input_tensor)
            end = time.perf_counter()
            latencies_ms.append((end - start) * 1000.0)
        overall_end = time.perf_counter()
        latencies_ms = np.asarray(latencies_ms)
        total_wall_time_s = overall_end - overall_start

    # =====================================================
    # Energy calculation
    # =====================================================
    average_power_w = None
    energy_per_image_mj = None
    if DEVICE.type == "cuda" and NVML_AVAILABLE and len(power_samples_mw) > 0:
        average_power_mw = float(np.mean(power_samples_mw))
        average_power_w = average_power_mw / 1000.0

        # Energy per image (mJ) = average power (W) x wall time (s) x 1000 / iterations
        energy_per_image_mj = average_power_w * total_wall_time_s * 1000.0 / timed_iters
    return {
        "mean_ms": float(np.mean(latencies_ms)),
        "median_ms": float(np.median(latencies_ms)),
        "p95_ms": float(np.percentile(latencies_ms, 95)),
        "min_ms": float(np.min(latencies_ms)),
        "max_ms": float(np.max(latencies_ms)),
        "average_power_w": average_power_w,
        "energy_per_image_mj": energy_per_image_mj,
    }

## 8. Peak GPU memory

In [35]:
def benchmark_peak_memory(inference_fn, input_tensor):
    # CPU has no CUDA memory statistic
    if DEVICE.type != "cuda":
        return {"baseline_mb": None, "peak_mb": None, "additional_peak_mb": None}

    # Clear unused cached memory
    torch.cuda.empty_cache()
    synchronize()

    # Baseline model/input memory
    baseline_bytes = torch.cuda.memory_allocated()

    # Reset peak tracker
    torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode():
        inference_fn(input_tensor)
    synchronize()
    peak_bytes = torch.cuda.max_memory_allocated()
    baseline_mb = baseline_bytes / (1024**2)
    peak_mb = peak_bytes / (1024**2)
    additional_peak_mb = max(0.0, peak_mb - baseline_mb)
    return {
        "baseline_mb": baseline_mb,
        "peak_mb": peak_mb,
        "additional_peak_mb": additional_peak_mb,
    }

## 9. Model and UEP initialisation

Builds each model and a UEP sized to its shared feature map.

In [36]:
MODEL_CLASSES = {
    "ResNet18": EarlyExitResNet,
    "VGG16": EarlyExitVGG,
    "MobileNetV2": EarlyExitMobileNet,
}

models_and_predictors = {}

for model_name, model_class in MODEL_CLASSES.items():
    print("\n" + "=" * 80)
    print(f"INITIALISING {model_name}")
    print("=" * 80)

    # -----------------------------------------------------
    # Randomly initialized main model
    # -----------------------------------------------------
    model = model_class(num_classes=NUM_CLASSES).to(DEVICE)
    model.eval()

    # -----------------------------------------------------
    # Determine feature-map dimensions
    # -----------------------------------------------------
    with torch.inference_mode():
        outputs, features = model(random_input)
    num_exits = len(outputs)
    input_channels = features.shape[1]
    spatial_dim = features.shape[2]

    # -----------------------------------------------------
    # Randomly initialized UEP
    # -----------------------------------------------------
    predictor = ExitPredictorV2(
        input_channels=input_channels, spatial_dim=spatial_dim, num_exits=num_exits
    ).to(DEVICE)
    predictor.eval()
    models_and_predictors[model_name] = (model, predictor)

    # -----------------------------------------------------
    # Print initialization information
    # -----------------------------------------------------
    print("Number of exits:", num_exits)
    print("Feature map:", tuple(features.shape))
    print("Main model parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e6:.3f} M")
    print("UEP parameters:", f"{sum(p.numel() for p in predictor.parameters()) / 1e6:.3f} M")


INITIALISING ResNet18
Number of exits: 2
Feature map: (1, 64, 56, 56)
Main model parameters: 11.184 M
UEP parameters: 0.425 M

INITIALISING VGG16
Number of exits: 3
Feature map: (1, 128, 56, 56)
Main model parameters: 134.309 M
UEP parameters: 0.444 M

INITIALISING MobileNetV2
Number of exits: 2
Feature map: (1, 24, 56, 56)
Main model parameters: 2.238 M
UEP parameters: 0.413 M


## 10. Profile every exit

For each model and exit, measures the backbone path alone and the UEP-inclusive path.

In [37]:
results = []

for model_name, (model, predictor) in models_and_predictors.items():
    print("\n\n")
    print("#" * 100)
    print(f"PROFILING {model_name}")
    print("#" * 100)

    # -----------------------------------------------------
    # Determine exits
    # -----------------------------------------------------
    with torch.inference_mode():
        outputs, _ = model(random_input)
    num_exits = len(outputs)

    # -----------------------------------------------------
    # Complete deployed parameter count
    # -----------------------------------------------------
    parameter_info = calculate_parameter_metrics(model, predictor, 0)
    total_deployed_params = parameter_info["Total_Deployed_Params"]
    main_model_params = parameter_info["Main_Model_Params"]
    uep_params = parameter_info["UEP_Params"]

    # =====================================================
    # Profile each exit
    # =====================================================
    for exit_idx in range(num_exits):
        exit_number = exit_idx + 1
        print("\n")
        print("-" * 100)
        print(f"{model_name} — EXIT {exit_number}")
        print("-" * 100)

        # =================================================
        # Parameter metrics
        # =================================================
        p = calculate_parameter_metrics(model, predictor, exit_idx)

        # =================================================
        # GFLOPs
        # =================================================
        main_gflops = calculate_gflops(model, predictor, random_input, exit_idx, include_uep=False)
        system_gflops = calculate_gflops(model, predictor, random_input, exit_idx, include_uep=True)

        # =================================================
        # Main model latency
        # =================================================
        main_latency = benchmark_latency(
            lambda x, idx=exit_idx: forward_to_exit(model, x, idx), random_input
        )

        # =================================================
        # UEP system latency
        # =================================================
        system_latency = benchmark_latency(
            lambda x, idx=exit_idx: forward_with_uep(model, predictor, x, idx), random_input
        )

        # =================================================
        # Main model peak memory
        # =================================================
        main_memory = benchmark_peak_memory(
            lambda x, idx=exit_idx: forward_to_exit(model, x, idx), random_input
        )

        # =================================================
        # UEP-inclusive peak memory
        # =================================================
        system_memory = benchmark_peak_memory(
            lambda x, idx=exit_idx: forward_with_uep(model, predictor, x, idx), random_input
        )

        # =================================================
        # Print metrics
        # =================================================
        print("\nPARAMETERS")
        print(f"Total deployed parameters : {total_deployed_params / 1e6:.3f} M")
        print(f"Main model parameters     : {main_model_params / 1e6:.3f} M")
        print(f"UEP parameters             : {uep_params / 1e6:.3f} M")
        print(f"Evaluated main parameters : {p['Evaluated_Main_Params'] / 1e6:.3f} M")
        print(f"Evaluated system parameters: {p['Evaluated_System_Params'] / 1e6:.3f} M")
        print(f"Evaluated fraction of deployed model: {p['Evaluated_Fraction_pct']:.2f}%")
        print("\nCOMPUTATION")
        print(f"Main path GFLOPs          : {main_gflops:.4f}")
        print(f"UEP-inclusive GFLOPs      : {system_gflops:.4f}")
        print("\nLATENCY")
        print(f"Main path mean latency    : {main_latency['mean_ms']:.3f} ms")
        print(f"Main path median latency  : {main_latency['median_ms']:.3f} ms")
        print(f"Main path P95 latency    : {main_latency['p95_ms']:.3f} ms")
        print(f"UEP system mean latency  : {system_latency['mean_ms']:.3f} ms")
        print(f"UEP system median latency: {system_latency['median_ms']:.3f} ms")
        print(f"UEP system P95 latency  : {system_latency['p95_ms']:.3f} ms")
        print("\nMEMORY")
        if main_memory["peak_mb"] is not None:
            print(f"Main peak GPU memory     : {main_memory['peak_mb']:.2f} MB")
            print(
                f"Main additional inference memory: {main_memory['additional_peak_mb']:.2f} MB"
            )
            print(f"UEP system peak GPU memory: {system_memory['peak_mb']:.2f} MB")
            print(
                f"UEP additional inference memory: {system_memory['additional_peak_mb']:.2f} MB"
            )
        else:
            print("GPU memory metrics: N/A (CPU)")
        print("\nENERGY")
        if system_latency["energy_per_image_mj"] is not None:
            print(f"Average GPU power        : {system_latency['average_power_w']:.2f} W")
            print(f"GPU energy / image       : {system_latency['energy_per_image_mj']:.3f} mJ")
        else:
            print("GPU energy / image       : N/A")

        # =================================================
        # Store results
        # =================================================
        results.append(
            {
                "Model": model_name,
                "Exit": f"Exit {exit_number}",
                # -------------------------------
                # Parameters
                # -------------------------------
                "Total_Deployed_Params_M": total_deployed_params / 1e6,
                "Main_Model_Params_M": main_model_params / 1e6,
                "UEP_Params_M": uep_params / 1e6,
                "Evaluated_Main_Params_M": p["Evaluated_Main_Params"] / 1e6,
                "Evaluated_System_Params_M": p["Evaluated_System_Params"] / 1e6,
                "Evaluated_Fraction_pct": p["Evaluated_Fraction_pct"],
                # -------------------------------
                # FLOPs
                # -------------------------------
                "Main_GFLOPs": main_gflops,
                "UEP_System_GFLOPs": system_gflops,
                # -------------------------------
                # Main latency
                # -------------------------------
                "Main_Mean_Latency_ms": main_latency["mean_ms"],
                "Main_Median_Latency_ms": main_latency["median_ms"],
                "Main_P95_Latency_ms": main_latency["p95_ms"],
                # -------------------------------
                # UEP system latency
                # -------------------------------
                "UEP_System_Mean_Latency_ms": system_latency["mean_ms"],
                "UEP_System_Median_Latency_ms": system_latency["median_ms"],
                "UEP_System_P95_Latency_ms": system_latency["p95_ms"],
                # -------------------------------
                # Memory
                # -------------------------------
                "Main_Peak_GPU_Memory_MB": main_memory["peak_mb"],
                "Main_Additional_Memory_MB": main_memory["additional_peak_mb"],
                "UEP_System_Peak_GPU_Memory_MB": system_memory["peak_mb"],
                "UEP_System_Additional_Memory_MB": system_memory["additional_peak_mb"],
                # -------------------------------
                # Energy
                # -------------------------------
                "Average_GPU_Power_W": system_latency["average_power_w"],
                "GPU_Energy_per_Image_mJ": system_latency["energy_per_image_mj"],
            }
        )

The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
model.final_classifier, model.final_classifier.1, model.resnet.avgpool, model.resnet.fc, model.resnet.layer4, model.resnet.layer4.0, model.resnet.layer4.0.bn1, model.resnet.layer4.0.bn2, model.resnet.layer4.0.conv1, model.resnet.layer4.0.conv2, model.resnet.layer4.0.downsample, model.resnet.layer4.0.downsample.0, model.resnet.layer4.0.downsample.1, model.resnet.layer4.0.relu, model.resnet.layer4.1, model.resnet.layer4.1.bn1, model.resnet.layer4.1.bn2, model.resnet.layer4.1.conv1, model.resnet.layer4.1.conv2, model.resnet.layer4.1.relu, predictor, predictor.conv_layers, predictor.conv_layers.0, predictor.conv_layers.1, predictor.conv_layers.2, predictor.conv_layers.3, pr




####################################################################################################
PROFILING ResNet18
####################################################################################################


----------------------------------------------------------------------------------------------------
ResNet18 — EXIT 1
----------------------------------------------------------------------------------------------------


The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
model.final_classifier, model.final_classifier.1, model.resnet.avgpool, model.resnet.fc, model.resnet.layer4, model.resnet.layer4.0, model.resnet.layer4.0.bn1, model.resnet.layer4.0.bn2, model.resnet.layer4.0.conv1, model.resnet.layer4.0.conv2, model.resnet.layer4.0.downsample, model.resnet.layer4.0.downsample.0, model.resnet.layer4.0.downsample.1, model.resnet.layer4.0.relu, model.resnet.layer4.1, model.resnet.layer4.1.bn1, model.resnet.layer4.1.bn2, model.resnet.layer4.1.conv1, model.resnet.layer4.1.conv2, model.resnet.layer4.1.relu, predictor, predictor.conv_layers, predictor.conv_layers.0, predictor.conv_layers.1, predictor.conv_layers.2, predictor.conv_layers.3, pr


PARAMETERS
Total deployed parameters : 11.609 M
Main model parameters     : 11.184 M
UEP parameters             : 0.425 M
Evaluated main parameters : 2.785 M
Evaluated system parameters: 3.210 M
Evaluated fraction of deployed model: 27.65%

COMPUTATION
Main path GFLOPs          : 1.4073
UEP-inclusive GFLOPs      : 1.4073

LATENCY
Main path mean latency    : 2.038 ms
Main path median latency  : 2.210 ms
Main path P95 latency    : 2.237 ms
UEP system mean latency  : 2.459 ms
UEP system median latency: 2.428 ms
UEP system P95 latency  : 2.832 ms

MEMORY
Main peak GPU memory     : 649.40 MB
Main additional inference memory: 7.97 MB
UEP system peak GPU memory: 649.40 MB
UEP additional inference memory: 7.97 MB

ENERGY
Average GPU power        : 50.35 W
GPU energy / image       : 125.821 mJ


----------------------------------------------------------------------------------------------------
ResNet18 — EXIT 2
----------------------------------------------------------------------------------

The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
model.block2, model.block3, model.exit2, model.exit2.0, model.exit2.1, model.exit2.2, model.final_classifier, model.final_classifier.1, model.vgg.avgpool, model.vgg.classifier, model.vgg.classifier.0, model.vgg.classifier.1, model.vgg.classifier.2, model.vgg.classifier.3, model.vgg.classifier.4, model.vgg.classifier.5, model.vgg.classifier.6, model.vgg.features.17, model.vgg.features.18, model.vgg.features.19, model.vgg.features.20, model.vgg.features.21, model.vgg.features.22, model.vgg.features.23, model.vgg.features.24, model.vgg.features.25, model.vgg.features.26, model.vgg.features.27, model.vgg.features.28, model.vgg.features.29, model.vgg.features.30, predictor, 

...


PARAMETERS
Total deployed parameters : 134.753 M
Main model parameters     : 134.309 M
UEP parameters             : 0.444 M
Evaluated main parameters : 134.302 M
Evaluated system parameters: 134.745 M
Evaluated fraction of deployed model: 99.99%

COMPUTATION
Main path GFLOPs          : 15.4662
UEP-inclusive GFLOPs      : 15.4662

LATENCY
Main path mean latency    : 9.007 ms
Main path median latency  : 9.005 ms
Main path P95 latency    : 9.143 ms
UEP system mean latency  : 9.131 ms
UEP system median latency: 9.136 ms
UEP system P95 latency  : 9.262 ms

MEMORY
Main peak GPU memory     : 672.89 MB
Main additional inference memory: 30.36 MB
UEP system peak GPU memory: 672.89 MB
UEP additional inference memory: 30.36 MB

ENERGY
Average GPU power        : 67.69 W
GPU energy / image       : 621.933 mJ



####################################################################################################
PROFILING MobileNetV2
###################################################################

The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
model.block2, model.mobilenet.classifier, model.mobilenet.classifier.0, model.mobilenet.classifier.1, model.mobilenet.features.14, model.mobilenet.features.14.conv, model.mobilenet.features.14.conv.0, model.mobilenet.features.14.conv.0.0, model.mobilenet.features.14.conv.0.1, model.mobilenet.features.14.conv.0.2, model.mobilenet.features.14.conv.1, model.mobilenet.features.14.conv.1.0, model.mobilenet.features.14.conv.1.1, model.mobilenet.features.14.conv.1.2, model.mobilenet.features.14.conv.2, model.mobilenet.features.14.conv.3, model.mobilenet.features.15, model.mobilenet.features.15.conv, model.mobilenet.features.15.conv.0, model.mobilenet.features.15.conv.0.0, mode


PARAMETERS
Total deployed parameters : 2.651 M
Main model parameters     : 2.238 M
UEP parameters             : 0.413 M
Evaluated main parameters : 0.543 M
Evaluated system parameters: 0.957 M
Evaluated fraction of deployed model: 36.10%

COMPUTATION
Main path GFLOPs          : 0.2222
UEP-inclusive GFLOPs      : 0.2222

LATENCY
Main path mean latency    : 3.687 ms
Main path median latency  : 3.645 ms
Main path P95 latency    : 3.964 ms
UEP system mean latency  : 4.134 ms
UEP system median latency: 4.100 ms
UEP system P95 latency  : 4.435 ms

MEMORY
Main peak GPU memory     : 650.90 MB
Main additional inference memory: 9.95 MB
UEP system peak GPU memory: 650.90 MB
UEP additional inference memory: 9.95 MB

ENERGY
Average GPU power        : 47.72 W
GPU energy / image       : 199.024 mJ


----------------------------------------------------------------------------------------------------
MobileNetV2 — EXIT 2
---------------------------------------------------------------------------------

The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
model.exit1, model.exit1.0, model.exit1.1, model.exit1.2, predictor, predictor.conv_layers, predictor.conv_layers.0, predictor.conv_layers.1, predictor.conv_layers.2, predictor.conv_layers.3, predictor.conv_layers.4, predictor.conv_layers.5, predictor.conv_layers.6, predictor.conv_layers.7, predictor.fc_layers, predictor.fc_layers.0, predictor.fc_layers.1, predictor.fc_layers.2, predictor.fc_layers.3, predictor.fc_layers.4
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though 


PARAMETERS
Total deployed parameters : 2.651 M
Main model parameters     : 2.238 M
UEP parameters             : 0.413 M
Evaluated main parameters : 2.237 M
Evaluated system parameters: 2.650 M
Evaluated fraction of deployed model: 99.96%

COMPUTATION
Main path GFLOPs          : 0.3129
UEP-inclusive GFLOPs      : 0.3129

LATENCY
Main path mean latency    : 5.098 ms
Main path median latency  : 5.007 ms
Main path P95 latency    : 5.811 ms
UEP system mean latency  : 5.474 ms
UEP system median latency: 5.375 ms
UEP system P95 latency  : 6.030 ms

MEMORY
Main peak GPU memory     : 650.90 MB
Main additional inference memory: 9.95 MB
UEP system peak GPU memory: 650.90 MB
UEP additional inference memory: 9.95 MB

ENERGY
Average GPU power        : 33.14 W
GPU energy / image       : 182.597 mJ


## 11. Results

Summary table; the full results are saved to CSV.

In [38]:
results_df = pd.DataFrame(results)

pd.set_option("display.max_columns", None)

pd.set_option("display.width", 300)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

reviewer_table = results_df[
    [
        "Model",
        "Exit",
        "Total_Deployed_Params_M",
        "Evaluated_System_Params_M",
        "Evaluated_Fraction_pct",
        "UEP_System_GFLOPs",
        "UEP_System_Median_Latency_ms",
        "UEP_System_P95_Latency_ms",
        "UEP_System_Peak_GPU_Memory_MB",
        "GPU_Energy_per_Image_mJ",
    ]
]

reviewer_table

,Model,Exit,Total_Deployed_Params_M,Evaluated_System_Params_M,Evaluated_Fraction_pct,UEP_System_GFLOPs,UEP_System_Median_Latency_ms,UEP_System_P95_Latency_ms,UEP_System_Peak_GPU_Memory_MB,GPU_Energy_per_Image_mJ
0,ResNet18,Exit 1,11.6092,3.2103,27.6534,1.4073,2.4278,2.8323,649.3970,125.8212
1,ResNet18,Exit 2,11.6092,11.6066,99.9779,1.8186,3.2452,3.5127,667.6694,223.7645
2,VGG16,Exit 1,134.7528,2.1816,1.6190,9.3353,4.1773,4.2377,667.4238,286.9874
3,VGG16,Exit 2,134.7528,8.0839,5.9991,13.9595,6.4118,6.5362,672.8945,441.4821
4,VGG16,Exit 3,134.7528,134.7451,99.9943,15.4662,9.1362,9.2616,672.8945,621.9330
5,MobileNetV2,Exit 1,2.6511,0.9570,36.0964,0.2222,4.1004,4.4346,650.8975,199.0238
6,MobileNetV2,Exit 2,2.6511,2.6501,99.9634,0.3129,5.3746,6.0295,650.8975,182.5972


In [39]:
OUTPUT_FILE = "UEPNet_random_weight_efficiency_metrics.csv"

results_df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)

Saved: UEPNet_random_weight_efficiency_metrics.csv
